# Data Feasibility

## tl;dr
Retrieved 1 outcome seasons and 16,752 canonical lineups. Internal reconciliation is distinct from independent source validation.

## Context & Methods
Compare sources before modeling. [Source recommendation](../docs/data_feasibility.md). [API documentation](https://api.pbpstats.com/docs).

### Key Assumptions
Regular-season PBP Stats aggregates; lagged profiles; observational comparisons. See [methodology](../docs/methodology_notes.md).

## Data
Run the cached pipeline and offline analysis before this notebook. No network requests are made here.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data_pipeline.py').exists())
DATA = ROOT / 'data' / 'processed'
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
pd.set_option('display.max_columns', 12)


## Results

### Source access tests

In [2]:
display(pd.read_json(ROOT/'docs'/'access_probes.json')[['name','seconds']])

,name,seconds
0,pbp_latest_teams,0.428975
1,nba_lineup,25.089524


### Season coverage

In [3]:
display(pd.read_csv(DATA/'season_summary.csv'))

,season,lineups,mean_side_possessions,minutes
0,2022-23,16752,244628.0,118970.0


### Team reconciliation

In [4]:
audit=pd.read_csv(ROOT/'docs'/'coverage_audit.csv')
cols=[c for c in audit if c.endswith('_difference')]
display(audit.groupby('season')[cols].agg(['min','max']))
display(audit.groupby('season').date_partitioned.sum().rename('capped_queries_recovered'))

SecondsPlayed_difference      OffPoss_difference       \
                             min  max                min  max   
season                                                          
2022-23                      0.0  0.0                0.0  0.0   

        DefPoss_difference      Points_difference       \
                       min  max               min  max   
season                                                   
2022-23                0.0  0.0               0.0  0.0   

        OpponentPoints_difference       
                              min  max  
season                                  
2022-23                       0.0  0.0

season
2022-23    27
Name: capped_queries_recovered, dtype: int64

### Validation evidence

In [5]:
display(json.loads((ROOT/'docs'/'validation_summary.json').read_text()))
display(pd.read_csv(ROOT/'docs'/'quarantine.csv'))

{'reconciliation': {'SecondsPlayed': {'max_absolute_difference': 0.0,
   'total_difference': 0.0},
  'OffPoss': {'max_absolute_difference': 0.0, 'total_difference': 0.0},
  'DefPoss': {'max_absolute_difference': 0.0, 'total_difference': 0.0},
  'Points': {'max_absolute_difference': 0.0, 'total_difference': 0.0},
  'OpponentPoints': {'max_absolute_difference': 0.0, 'total_difference': 0.0}},
 'lineups': 16752,
 'seasons': ['2022-23'],
 'zero_off_poss': 734,
 'zero_def_poss': 545,
 'nonfinite_net_rating': 1089}

,season,team_id,raw_id,reason


## Takeaways
The 500-row API cap required date partitioning. Preserve source conventions and caches. External NBA box-score reconciliation remains a research limitation.